# PlantDoctor - train the full 10-class tomato model (Kaggle version)

Same training pipeline as `train_tomato_model.ipynb`, adapted to run on Kaggle instead of Colab:

1. **Settings (right sidebar) > Accelerator > GPU T4 x2** (or P100) - free, no laptop GPU needed.
2. **Add Data** (right sidebar) and search for a PlantVillage tomato dataset - see next cell for which one.
3. Run all cells top to bottom - training happens here on Kaggle's GPU, then you download a small
   `model_export.zip` and convert it to TF.js **locally** in one last step (see the notebook's end
   for why - Kaggle's pre-installed packages fight with the `tensorflowjs` pip package).

## 1. Data

Click **Add Input** > search **`plantvillage dataset`** > add **"PlantVillage Dataset"** by `abdallahalidev`
(https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset). It mirrors the same public
PlantVillage data the Colab notebook sparse-clones from GitHub, already split into `color` /
`grayscale` / `segmented` folders with one subfolder per class - no download/clone step needed,
Kaggle mounts it read-only at `/kaggle/input/`.

The cell below finds the `Tomato___*` class folders under whatever the dataset's mount path turns
out to be, preferring the `color` variant (closest to real phone photos - `grayscale`/`segmented`
would train a model that doesn't match what `App.js` feeds it).

In [ ]:
import os, pathlib

# Plain rglob("Tomato___*") also descends INTO every match (and every other
# crop's class folder - Potato___*, Corn___*, ...) looking for further
# matches, which means listing every one of the ~50k+ image files in the
# dataset - that's the "stuck for minutes" cause. os.walk + pruning any
# "Crop___Disease"-shaped folder (PlantVillage's naming convention) keeps
# this to just the directory skeleton: seconds, not minutes.
tomato_dirs = set()
for root, dirnames, _files in os.walk("/kaggle/input"):
    keep = []
    for d in dirnames:
        if "___" in d:
            if d.startswith("Tomato___"):
                tomato_dirs.add(pathlib.Path(root))
            continue  # leaf class folder full of images - don't descend into it
        keep.append(d)
    dirnames[:] = keep

candidates = sorted(tomato_dirs)
assert candidates, "No Tomato___* folders found - did you add the PlantVillage dataset via 'Add Input'?"

color_dirs = [p for p in candidates if p.name == "color"]
DATA_DIR = color_dirs[0] if color_dirs else candidates[0]
print("Using:", DATA_DIR)

In [ ]:
classes = sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir() and d.name.startswith("Tomato___"))
print(len(classes), "classes:")
for c in classes:
    n = len(list((DATA_DIR / c).glob("*.JPG"))) + len(list((DATA_DIR / c).glob("*.jpg")))
    print(f"  {c}: {n} images")

You should see exactly 10 `Tomato___*` classes (healthy + 9 diseases), matching the classes in
`C:\Users\dell\Desktop\testing` used to validate this model.

## 2. Build train/val datasets

`image_dataset_from_directory` only reads the `Tomato___*` subfolders it's pointed at, so the
other crops elsewhere in the dataset are never touched.

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=123,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_names=classes)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=123,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_names=classes)

class_names = train_ds.class_names  # authoritative order used for the labels array
print(class_names)

# Normalize to [-1, 1] here in the data pipeline, matching exactly what
# App.js's classifyImage does (.div(127.5).sub(1)) before calling the model.
# The exported model must NOT also normalize internally - App.js already
# hands it normalized input, so baking preprocess_input into the model graph
# would double-normalize at inference (and also serializes as a raw TF op
# that doesn't survive an H5 round-trip across Keras versions).
def normalize(x, y):
    return x / 127.5 - 1.0, y

train_ds = train_ds.map(normalize).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(normalize).prefetch(tf.data.AUTOTUNE)
# No .cache(): caching ~18k decoded 224x224 images in RAM is what blew out
# Colab's free-tier memory; same risk on Kaggle's 13-16GB instances. Disk
# read + decode is cheap next to the GPU compute anyway.

## 3. Build the model (MobileNetV2 transfer learning)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base_model.trainable = False

# Named separately (not inline) so the export cell later can reuse these
# exact trained layer objects to build a clean inference-only graph.
pooling = tf.keras.layers.GlobalAveragePooling2D()
dropout = tf.keras.layers.Dropout(0.2)
classifier = tf.keras.layers.Dense(len(class_names), activation="softmax")

# Training graph only - includes augmentation (helps generalization, and is
# a no-op at inference time anyway). Input here is already normalized to
# [-1, 1] by the tf.data pipeline above, matching what App.js hands the
# model, so no preprocessing layer is needed in the graph.
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = pooling(x)
x = dropout(x)
outputs = classifier(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Train (frozen base, fast)

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=8)

## 5. Optional: fine-tune the top of MobileNetV2

Usually improves accuracy a few points. Skip this cell if step 4's validation accuracy is already
good enough.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=5)

## 6. Evaluate

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f"Validation accuracy: {acc:.3f}")

## 7. Save the trained model

We stop here on Kaggle. Converting to TF.js needs the `tensorflowjs` pip package, and Kaggle's
base image pre-installs `tensorflow`/`tf_keras`/`tensorflow_decision_forests` at versions that
fight with it no matter how it's installed (global pip, `venv`, `virtualenv` all hit a different
broken import each time - numpy aliases, a TF-DF ABI mismatch, a `tf_keras` circular import,
`ensurepip` missing). None of that is a GPU/disk problem, so there's no reason to fight it here -
the conversion step is small and fast enough to just do it locally after downloading the model.

In [ ]:
# Build a clean inference-only model reusing the trained layer objects
# (base_model, pooling, dropout, classifier - same weights as `model`) but
# WITHOUT data_augmentation and without any in-graph preprocessing. App.js
# normalizes pixels to [-1, 1] itself before calling the model, so the
# exported model must take that already-normalized input directly - and
# dropping augmentation/raw-op layers also keeps this a plain, portable
# Keras graph (Input -> MobileNetV2 -> pool -> dropout -> dense) that
# survives an H5 round-trip and converts to TF.js cleanly.
inference_inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input")
x = base_model(inference_inputs, training=False)
x = pooling(x)
x = dropout(x, training=False)
outputs = classifier(x)
inference_model = tf.keras.Model(inference_inputs, outputs)

inference_model.save("/kaggle/working/model.h5")

import json
metadata = {
    "modelName": "plantdoctor-tomato-10class",
    "labels": class_names,
    "imageSize": IMG_SIZE,
}
with open("/kaggle/working/metadata.json", "w") as f:
    json.dump(metadata, f)

print("Saved:")
!ls -la /kaggle/working/model.h5 /kaggle/working/metadata.json

In [ ]:
!cd /kaggle/working && zip -q model_export.zip model.h5 metadata.json
print("Done - open the notebook's Output pane (after Save Version) and download model_export.zip")

## 8. Download

**Save Version > Save & Run All (Commit)**, wait for it to finish, then open that version's
**Output** tab and download `model_export.zip` (~10-20MB - just the model weights and a small
metadata file, no dataset).

## 9. Convert to TF.js locally (on your laptop)

This needs no GPU and barely any disk - just a throwaway Python env to run the converter once.
Unzip `model_export.zip` somewhere, then from that folder in PowerShell:

```powershell
python -m venv tfjs_env
tfjs_env\Scripts\activate
pip install tensorflowjs
tensorflowjs_converter --input_format=keras model.h5 tfjs_model
```

That produces a `tfjs_model\` folder with `model.json` + one or more `group1-shard*.bin` files.
Copy `metadata.json` (from the zip) into that same folder too.

## 10. Install into PlantDoctor

Replace everything in `public/model/` in this repo with the three files: `model.json`,
`group1-shard*.bin`, and `metadata.json`. No app code changes needed - `App.js` reads
`metadata.json`'s `labels` array and resolves weight file paths from whatever `model.json` says.
Then `npm run web` and spot-check a few images per class from `C:\Users\dell\Desktop\testing\`.